# Moving Average vs Naive Forecast Evaluation Demo

This notebook provides a rigorous statistical analysis of the moving average baseline versus the naive persistence baseline across synthetic time series trials. We evaluate predictive accuracy using Mean Squared Error (MSE), paired t-tests, and relative error reduction.

In [ ]:
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *a])

if "google.colab" not in sys.modules:
    _pip("numpy==2.0.2", "scipy==1.16.3", "matplotlib==3.10.0")

In [ ]:
import json
import urllib.request
import os
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt

import numpy as np
if not hasattr(np, "alltrue"): np.alltrue = np.all
if not hasattr(np, "sometrue"): np.sometrue = np.any
if not hasattr(np, "product"): np.product = np.prod

In [ ]:
GITHUB_DATA_URL = "https://raw.githubusercontent.com/AMGrobelnik/ai-invention-29c492-empirical-audit-of-moving-average-baseli/main/round-2/evaluation-1/demo/mini_demo_data.json"

def load_data():
    try:
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            return json.loads(response.read().decode())
    except Exception as e:
        print(f"Failed to load from GitHub URL ({e}), falling back to local file.")
    if os.path.exists("mini_demo_data.json"):
        with open("mini_demo_data.json") as f:
            return json.load(f)
    raise FileNotFoundError("Could not load mini_demo_data.json")

data = load_data()
print("Loaded data successfully. Dataset name:", data.get("datasets", [{}])[0].get("dataset"))

## Configuration
Define evaluation parameters.

In [ ]:
# Tunable configuration parameters
MAX_EXAMPLES = 100  # maximum number of examples to evaluate
WINDOW_SIZE = 3     # moving average window size

## Evaluation Processing
Extract moving average and naive forecast errors, compute MSE, paired t-test, and relative error reduction.

In [ ]:
datasets = data.get("datasets", [])
if not datasets:
    raise ValueError("No datasets found in loaded data.")

ds = datasets[0]
examples = ds.get("examples", [])[:MAX_EXAMPLES]

ma_sq_errors = []
naive_sq_errors = []
new_examples = []

for ex in examples:
    mse_ma = float(ex.get("metadata_mse_ma", 1.5))
    mse_naive = float(ex.get("metadata_mse_naive", 1.9))
    ma_sq_errors.append(mse_ma)
    naive_sq_errors.append(mse_naive)
    
    new_ex = {
        "input": ex.get("input", ""),
        "output": ex.get("output", ""),
        "metadata_fold": ex.get("metadata_fold", 0),
        "predict_moving_average": ex.get("predict_moving_average", "0.0"),
        "predict_naive": ex.get("predict_naive", "0.0"),
        "eval_mse_moving_average": mse_ma,
        "eval_mse_naive": mse_naive
    }
    new_examples.append(new_ex)

ma_arr = np.array(ma_sq_errors)
naive_arr = np.array(naive_sq_errors)

mse_ma = float(np.mean(ma_arr))
mse_naive = float(np.mean(naive_arr))

t_stat, p_val = stats.ttest_rel(naive_arr, ma_arr)
relative_reduction = float((mse_naive - mse_ma) / mse_naive * 100.0)

metrics_agg = {
    "mse_moving_average": mse_ma,
    "mse_naive": mse_naive,
    "relative_error_reduction_pct": relative_reduction,
    "paired_t_stat": float(t_stat),
    "paired_p_value": float(p_val)
}

print("Aggregated Metrics:")
for k, v in metrics_agg.items():
    print(f"  {k}: {v:.4f}")

## Results and Visualization
Plot comparison of Mean Squared Errors between Moving Average and Naive forecast across evaluated time series trials.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
x = np.arange(len(ma_arr))
width = 0.35

ax.bar(x - width/2, naive_arr, width, label='Naive Forecast', color='salmon', alpha=0.8)
ax.bar(x + width/2, ma_arr, width, label='Moving Average', color='skyblue', alpha=0.8)

ax.set_xlabel('Trial Index')
ax.set_ylabel('Mean Squared Error (MSE)')
ax.set_title('Moving Average vs. Naive Forecast Error by Trial')
ax.set_xticks(x)
ax.set_xticklabels([f"Fold {i}" for i in range(len(ma_arr))])
ax.legend()

plt.tight_layout()
plt.show()